In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from frechetdist import frdist
from keras.preprocessing.sequence import pad_sequences
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
from scipy.spatial.distance import directed_hausdorff
from scipy.stats import gaussian_kde
import seaborn as sns
from shapely.geometry import Polygon
from sklearn.metrics import auc, roc_curve, accuracy_score
from sklearn.metrics.pairwise import cosine_similarity
import tensorflow as tf
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

In [3]:
import keras
print(keras.__version__)

2.2.4


In [4]:
import sys
print(sys.version)

3.6.8 (tags/v3.6.8:3c6b436a57, Dec 24 2018, 00:16:47) [MSC v.1916 64 bit (AMD64)]


In [5]:
import os
import sys

# Use this to load our modules.
module_path = os.path.abspath(os.path.join(os.getcwd(), '../../LSTM-TrajGAN'))
if module_path not in sys.path:
    sys.path.append(module_path)

from model import LSTM_TrajGAN
from data.csv2npynew import data_conversion 

In [6]:
SEED = 2023
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.compat.v1.set_random_seed(SEED)

# Utils functions

In [7]:
def data_conversion(df, tid_col,test=False):
    """Converts input panda dataframe to one-hot-encoded Numpy array (locations are still in float)."""
    
    x = [[] for i in ['lat_lon', 'day', 'hour', 'category', 'mask','tid','count']]
    for tid in df[tid_col].unique():
        traj = df.loc[df[tid_col].isin([tid])]
        features = np.transpose(traj.loc[:, ['lat', 'lon', 'day', 'hour', 'category']].values)
        loc_list = []
        for i in range(0, len(traj)):
            lat = traj['lat'].values[i]
            lon = traj['lon'].values[i]
            loc_list.append(np.array([lat, lon], dtype=np.float64))
        x[0].append(loc_list)
        x[1].append(np.eye(7)[features[2].astype(np.int32)])
        x[2].append(np.eye(24)[features[3].astype(np.int32)])
        x[3].append(np.eye(10)[features[4].astype(np.int32)])
        x[4].append(np.ones(shape=(features[0].shape[0],1))) # mask
        
        if test:
            x[5].append(tid)
            x[6].append(traj.tid.count())
        

    converted_data = np.array([np.array(f, dtype=object) for f in x])
    return converted_data

def data_deconversion(x_test, x_prediction, df_test, max_length):
    """
    Converts one-hot-encoded Numpy array prediction (locations are still in float) to pd dataframe.

    x_test: original test set
    x_prediction: analog synthetic set
    
    """
    traj_attr_concat_list = []
    for attributes in x_prediction:
        traj_attr_list = []
        idx = 0
        for row in attributes:
            if row.shape == (max_length, 2): #If we have lat , lon
                traj_attr_list.append(row[max_length-x_test[6][idx][0]:])
            else:
                traj_attr_list.append(np.argmax(row[max_length-x_test[6][idx][0]:],axis=1).reshape(x_test[6][idx][0],1))
            idx += 1
        traj_attr_concat = np.concatenate(traj_attr_list)
        traj_attr_concat_list.append(traj_attr_concat)
    traj_data = np.concatenate(traj_attr_concat_list,axis=1)

    label = np.array(df_test['label']).reshape(-1,1)
    tid = np.array(df_test['tid']).reshape(-1,1)
    traj_data = np.concatenate([label,tid,traj_data],axis=1)
    df_traj_fin = pd.DataFrame(traj_data)
        
    df_traj_fin.columns = ['label','tid','lat','lon','day', 'hour', 'category','mask']    
        
    del df_traj_fin['mask']
        
    df_traj_fin['tid'] = df_traj_fin['tid'].astype(np.int32)
    df_traj_fin['day'] = df_traj_fin['day'].astype(np.int32)
    df_traj_fin['hour'] = df_traj_fin['hour'].astype(np.int32)
    df_traj_fin['category'] = df_traj_fin['category'].astype(np.int32)
    df_traj_fin['label'] = df_traj_fin['label'].astype(np.int32)

    return df_traj_fin

def scale(df_origin, GAN):
    df = df_origin.copy()
    lat_centroid = GAN.lat_centroid
    lon_centroid = GAN.lon_centroid

    scale_factor = GAN.scale_factor

    df["lat"] = df["lat"].apply(lambda x : x - lat_centroid)
    df["lon"] = df["lon"].apply(lambda x : x - lon_centroid)
    return df

def unscale(df, GAN):
    df_unscaled = df.copy()
    df_unscaled['lat'] = df_unscaled['lat'] + GAN.lat_centroid
    df_unscaled['lon'] = df_unscaled['lon'] + GAN.lon_centroid
    return df_unscaled

def Dist_1_1(test: pd.DataFrame, syn: pd.DataFrame, distance: str):
    syn_dist = []

    assert test["label"].unique().tolist() == syn["label"].unique().tolist() , "Users in test and syn aren't the same"

    for label in test["label"].unique():

        tidlist = list(test[test["label"] == label ]['tid'].unique())
        for  tid in  tidlist:

            real_traj = np.concatenate([np.array(test.loc[test['tid'] == tid]["lat"]).reshape(-1,1),
                                        np.array(test.loc[test['tid'] == tid]["lon"]).reshape(-1,1)],
                                    axis=1)
            syn_traj = np.concatenate([np.array(syn.loc[syn['tid'] == tid]["lat"]).reshape(-1,1),
                                    np.array(syn.loc[syn['tid'] == tid]["lon"]).reshape(-1,1)],
                                    axis=1)

            if distance == 'Haussdorff':
                syn_dist.append(directed_hausdorff(real_traj,syn_traj)[0])
            elif distance == 'Frechet':
                syn_dist.append(frdist(real_traj,syn_traj))

    return syn_Haus


def hours_1_1(test: pd.DataFrame, syn: pd.DataFrame):
    dis_cos = []

    assert test["label"].unique().tolist() == syn["label"].unique().tolist() , "Users in test and syn aren't the same"

    for label in test["label"].unique():

        tidlist = list(test[test["label"] == label ]['tid'].unique())
        
        for  tid in  tidlist:

            real_hour = np.array(test.loc[test['tid'] == tid]["hour"]).reshape(-1,1)
            syn_hour = np.array(syn.loc[syn['tid'] == tid]["hour"]).reshape(-1,1)


            dis_cos.append(np.linalg.norm(real_hour - syn_hour))
        
    return dis_cos 

def cat_1_1(test: pd.DataFrame, syn: pd.DataFrame):
    dis_cos = []

    assert test["label"].unique().tolist() == syn["label"].unique().tolist() , "Users in test and syn aren't the same"

    for label in test["label"].unique():

        tidlist = list(test[test["label"] == label ]['tid'].unique())
        
        for  tid in  tidlist:

            real_day = np.array(test.loc[test['tid'] == tid]["day"]).reshape(-1,1)
            syn_day = np.array(syn.loc[syn['tid'] == tid]["day"]).reshape(-1,1)
            
            real_cat = np.array(test.loc[test['tid'] == tid]["category"]).reshape(-1,1)
            syn_cat = np.array(syn.loc[syn['tid'] == tid]["category"]).reshape(-1,1)


            dis_cos.append(np.linalg.norm(real_day - syn_day) + np.linalg.norm(real_cat - syn_cat))
        
    return dis_cos 

def find_closest_cat(df_syn, trajectory):
    # trajectory does not contain any tid and label column
    dis_euclide = []
    tid_min = None
    mini = 99
    for label in df_syn['label'].unique():
        for tid in df_syn[df_syn['label']==label]['tid'].unique():
            syn_day = np.array(df_syn.loc[df_syn['tid'] == tid]["day"]).reshape(-1,1)
            syn_category = np.array(df_syn.loc[df_syn['tid'] == tid]["category"]).reshape(-1,1)

            if len(syn_day) == len(np.array(trajectory["day"]).reshape(-1,1)) and len(syn_category) == len(np.array(trajectory["category"]).reshape(-1,1)) :
                tamp = (np.linalg.norm(np.array(trajectory["day"]).reshape(-1,1) - syn_day) + np.linalg.norm(np.array(trajectory["category"]).reshape(-1,1) - syn_category))
                if mini >= tamp:
                    mini = tamp
                    tid_min = tid
    return tid_min, mini

def find_closest_traj(df_syn, trajectory):
    # trajectory does not contain any tid and label column
    dis_euclide = []
    tid_min = None
    mini = 99
    tamp_lat = np.array(trajectory["lat"]).reshape(-1,1)
    tamp_lon = np.array(trajectory["lon"]).reshape(-1,1)
    real_traj = np.concatenate([tamp_lat, tamp_lon], axis= 1)
    for label in df_syn['label'].unique():
        for tid in df_syn[df_syn['label']==label]['tid'].unique():
            syn_lat = np.array(df_syn.loc[df_syn['tid'] == tid]["lat"]).reshape(-1,1)
            syn_lon = np.array(df_syn.loc[df_syn['tid'] == tid]["lon"]).reshape(-1,1)

            if len(syn_lat) == len(tamp_lat) :
                syn_traj = np.concatenate([syn_lat, syn_lon], axis=1)
                tamp = frdist(real_traj, syn_traj)
                if mini >= tamp:
                    mini = tamp
                    tid_min = tid
    return tid_min, mini

def matching(df_syn, df_real):
    count = 0
    for tid in tqdm(df_real['tid'].unique()):
        trajectory = df_real.loc[df_real['tid'] == tid]
        tid_min,_ = find_closest(df_syn, trajectory)
        if tid == tid_min:
            count += 1
    return count / df_real['tid'].nunique()

def synthesize(df, GAN):
    # Scale the trajectories
    df_scaled = scale(df, GAN)
    # Apply One-Hot encoding
    x_test = data_conversion(df_scaled, "tid", test=True)
    x_test = [x_test[0],x_test[1],x_test[2],x_test[3],x_test[4],x_test[5].reshape(-1,1),x_test[6].reshape(-1,1)]
    X_test = [pad_sequences(f, GAN.max_length, padding='pre', dtype='float64') for f in x_test[:5]]

    num_traj = X_test[0].shape[0]

    noise = np.random.normal(0, 1, (num_traj, 100))
    X_test.append(noise)

    prediction = GAN.generator.predict(X_test)
    
    # Reverse One-Hot encoding
    df_syn = data_deconversion(x_test, prediction, df_scaled, GAN.max_length)

    df_syn = unscale(df_syn, GAN)
    return df_syn

def find_closest_custom(df_syn, trajectory):
    distances = []
    tid_min = None
    mini = 99
    tamp_lat = np.array(trajectory["lat"]).reshape(-1,1)
    tamp_lon = np.array(trajectory["lon"]).reshape(-1,1)
    real_traj = np.concatenate([tamp_lat, tamp_lon], axis= 1)
    for label in df_syn['label'].unique():
        for tid in df_syn[df_syn['label']==label]['tid'].unique():
            syn_lat = np.array(df_syn.loc[df_syn['tid'] == tid]["lat"]).reshape(-1,1)
            syn_lon = np.array(df_syn.loc[df_syn['tid'] == tid]["lon"]).reshape(-1,1)
            syn_cat = np.array(df_syn.loc[df_syn['tid'] == tid]["category"]).reshape(1,-1)
            syn_hour = np.array(df_syn.loc[df_syn['tid'] == tid]["hour"]).reshape(1,-1)
            syn_day = np.array(df_syn.loc[df_syn['tid'] == tid]["day"]).reshape(1,-1)

            if len(syn_lat) == len(tamp_lat) :
                syn_traj = np.concatenate([syn_lat, syn_lon], axis=1)
                frech_dist = frdist(real_traj, syn_traj)
                cos_dist_cat = 1 - cosine_similarity(np.array(trajectory["category"]).reshape(1,-1), syn_cat)[0][0]
                cos_dist_hour = 1 - cosine_similarity(np.array(trajectory["hour"]).reshape(1,-1), syn_hour)[0][0]
                cos_dist_day = 1 - cosine_similarity(np.array(trajectory["day"]).reshape(1,-1), syn_day)[0][0]
                tamp = cos_dist_cat + cos_dist_hour + cos_dist_day + frech_dist
                if mini >= tamp:
                    mini = tamp
                    tid_min = tid
    return tid_min, mini

In [8]:
# global variables
PATH_TO_DATA_TRAIN = #Your data train path
PATH_TO_DATA_TEST = #Your data test path 
Frechet = False
Custom = True

# Paper results

## 1. Standard MIA

### a. Split the dataset (trajectory level)

In [9]:
print(f'------------------------------- SPLIT in the GLOBAL dataset ---------------------------------------------------------------')
train_dataset = pd.read_csv(PATH_TO_DATA_TRAIN)
test_dataset = pd.read_csv(PATH_TO_DATA_TEST)
full_dataset = pd.concat([train_dataset,test_dataset], axis=0)
print(f"num users in the full dataset: {full_dataset['label'].nunique()} and total num of trajectories: {full_dataset['tid'].nunique()}")

split_cat = 'tid'
ids = full_dataset[split_cat].unique()
np.random.shuffle(ids)
split = int(len(ids)*0.5)
df_target, df_aux = full_dataset[full_dataset[split_cat].isin(ids[:split])], full_dataset[full_dataset[split_cat].isin(ids[split:])]

print(f"num users in the dataset target: {df_target['label'].nunique()} and total num of trajectories in it: {df_target['tid'].nunique()}")
print(f"num users in the dataset auxiliary: {df_aux['label'].nunique()} and total num of trajectories in it: {df_aux['tid'].nunique()}")

print(f'------------------------------- SPLITS in the TARGET dataset ---------------------------------------------------------------')

ids = df_target[split_cat].unique()
np.random.shuffle(ids)
split1 = int(len(ids)*0.7) 
split2 = int(len(ids)*0.90) 
df_target_train, Q_target_gen, Q_target_test = df_target[df_target[split_cat].isin(ids[:split1])], df_target[df_target[split_cat].isin(ids[split1:split2])],df_target[df_target[split_cat].isin(ids[split2:])]

print(f"num users in D target Train: {df_target_train['label'].nunique()} and total num of trajectories in it: {df_target_train['tid'].nunique()}")
print(f"num users in Q target for synthetization: {Q_target_gen['label'].nunique()} and total num of trajectories in it: {Q_target_gen['tid'].nunique()}")
print(f"num users in Q target for tests: {Q_target_test['label'].nunique()} and total num of trajectories in it: {Q_target_test['tid'].nunique()}")

print(f'------------------------------- SPLITS in the AUXILIARY dataset ---------------------------------------------------------------')

ids = df_aux[split_cat].unique()
np.random.shuffle(ids)
split1 = int(len(ids)*0.7) 
split2 = int(len(ids)*0.90) 
df_aux_train, Q_aux_gen, Q_aux_test = df_aux[df_aux[split_cat].isin(ids[:split1])], df_aux[df_aux[split_cat].isin(ids[split1:split2])],df_aux[df_aux[split_cat].isin(ids[split2:])]

print(f"num users in D aux Train: {df_aux_train['label'].nunique()} and total num of trajectories in it: {df_aux_train['tid'].nunique()}")
print(f"num users in Q aux for synthetization: {Q_aux_gen['label'].nunique()} and total num of trajectories in it: {Q_aux_gen['tid'].nunique()}")
print(f"num users in Q aux for tests: {Q_aux_test['label'].nunique()} and total num of trajectories in it: {Q_aux_test['tid'].nunique()}")

------------------------------- SPLIT in the GLOBAL dataset ---------------------------------------------------------------
num users in the full dataset: 193 and total num of trajectories: 3079
num users in the dataset target: 193 and total num of trajectories in it: 1539
num users in the dataset auxiliary: 193 and total num of trajectories in it: 1540
------------------------------- SPLITS in the TARGET dataset ---------------------------------------------------------------
num users in D target Train: 193 and total num of trajectories in it: 1077
num users in Q target for synthetization: 149 and total num of trajectories in it: 308
num users in Q target for tests: 104 and total num of trajectories in it: 154
------------------------------- SPLITS in the AUXILIARY dataset ---------------------------------------------------------------
num users in D aux Train: 192 and total num of trajectories in it: 1078
num users in Q aux for synthetization: 158 and total num of trajectories in it:

### b. Train / load the models

In [10]:
# Global variables set by LSTMTrajGAN
latent_dim = 100
max_length = 144
GAN_PATH = # Your saving path

keys = ['lat_lon', 'day', 'hour', 'category', 'mask']
vocab_size = {"lat_lon":2,"day":7,"hour":24,"category":10,"mask":1}

n_epochs = 2000
n_batch_size = 128
n_sample_interval = 100

lat_centroid_aux = df_aux_train['lat'].sum()/len(df_aux_train)
lon_centroid_aux = df_aux_train['lon'].sum()/len(df_aux_train)

scale_factor_aux = max(max(abs(df_aux_train['lat'].max() - lat_centroid_aux),
                     abs(df_aux_train['lat'].min() - lat_centroid_aux),
                    ),
                 max(abs(df_aux_train['lon'].max() - lon_centroid_aux),
                     abs(df_aux_train['lon'].min() - lon_centroid_aux),
                    ))

gan_aux = LSTM_TrajGAN(latent_dim, keys, vocab_size, max_length, lat_centroid_aux, lon_centroid_aux, scale_factor_aux, SEED, all_params_path = GAN_PATH)
gan_aux_path = f'{GAN_PATH}\training_params\G_model_' + str(n_epochs) + '_paper_auxiliary.h5'

df_aux_train_normalized = df_aux_train.copy()
df_aux_train_normalized["lat"] = df_aux_train_normalized["lat"].apply(lambda x : x - lat_centroid_aux)
df_aux_train_normalized["lon"] = df_aux_train_normalized["lon"].apply(lambda x : x - lon_centroid_aux)
gan_aux.train(train=df_aux_train_normalized, epochs=n_epochs, batch_size=n_batch_size, sample_interval=n_sample_interval, adding = 'paper_auxiliary')

Instructions for updating:
Colocations handled automatically by placer.
Instructions for updating:
Use tf.cast instead.
[1/2000] D Loss: 32.344742 | G Loss: 1101.659302
[2/2000] D Loss: 30.709106 | G Loss: 1074.610352
[3/2000] D Loss: 29.054852 | G Loss: 1071.438721
[4/2000] D Loss: 27.368639 | G Loss: 1065.688843
[5/2000] D Loss: 25.718809 | G Loss: 1051.898315
[6/2000] D Loss: 24.126293 | G Loss: 1051.343140
[7/2000] D Loss: 22.585922 | G Loss: 1047.342407
[8/2000] D Loss: 21.109249 | G Loss: 1040.335938
[9/2000] D Loss: 19.694069 | G Loss: 1038.293701
[10/2000] D Loss: 18.351707 | G Loss: 1036.786621
[11/2000] D Loss: 17.062080 | G Loss: 1032.703247
[12/2000] D Loss: 15.834099 | G Loss: 1026.992920
[13/2000] D Loss: 14.672258 | G Loss: 1026.720093
[14/2000] D Loss: 13.569592 | G Loss: 1028.427124
[15/2000] D Loss: 12.531857 | G Loss: 1026.413940
[16/2000] D Loss: 11.545378 | G Loss: 1021.779480
[17/2000] D Loss: 10.618124 | G Loss: 1017.625305
[18/2000] D Loss: 9.738260 | G Loss: 10

In [11]:
# Global variables set by LSTMTrajGAN
latent_dim = 100
max_length = 144

keys = ['lat_lon', 'day', 'hour', 'category', 'mask']
vocab_size = {"lat_lon":2,"day":7,"hour":24,"category":10,"mask":1}

n_epochs = 2000
n_batch_size = 128
n_sample_interval = 100

lat_centroid_target = df_target_train['lat'].sum()/len(df_target_train)
lon_centroid_target = df_target_train['lon'].sum()/len(df_target_train)

scale_factor_target = max(max(abs(df_target_train['lat'].max() - lat_centroid_target),
                     abs(df_target_train['lat'].min() - lat_centroid_target),
                    ),
                 max(abs(df_target_train['lon'].max() - lon_centroid_target),
                     abs(df_target_train['lon'].min() - lon_centroid_target),
                    ))

gan_target = LSTM_TrajGAN(latent_dim, keys, vocab_size, max_length, lat_centroid_target, lon_centroid_target, scale_factor_target, SEED, all_params_path = GAN_PATH)
gan_target_path = f'{GAN_PATH}\training_params\G_model_' + str(n_epochs) + '_paper_target.h5'

df_target_train_normalized = df_target_train.copy()
df_target_train_normalized["lat"] = df_target_train_normalized["lat"].apply(lambda x : x - lat_centroid_target)
df_target_train_normalized["lon"] = df_target_train_normalized["lon"].apply(lambda x : x - lon_centroid_target)
gan_target.train(train=df_target_train_normalized, epochs=n_epochs, batch_size=n_batch_size, sample_interval=n_sample_interval, adding = 'paper_target')

[1/2000] D Loss: 32.344997 | G Loss: 1108.010498
[2/2000] D Loss: 30.710123 | G Loss: 1073.289551
[3/2000] D Loss: 29.054146 | G Loss: 1071.926880
[4/2000] D Loss: 27.371742 | G Loss: 1063.884766
[5/2000] D Loss: 25.718155 | G Loss: 1056.569824
[6/2000] D Loss: 24.124527 | G Loss: 1049.717773
[7/2000] D Loss: 22.582214 | G Loss: 1041.023682
[8/2000] D Loss: 21.107224 | G Loss: 1041.513916
[9/2000] D Loss: 19.689281 | G Loss: 1040.655396
[10/2000] D Loss: 18.348301 | G Loss: 1036.316040
[11/2000] D Loss: 17.064476 | G Loss: 1040.018921
[12/2000] D Loss: 15.833040 | G Loss: 1029.771729
[13/2000] D Loss: 14.672828 | G Loss: 1030.065063
[14/2000] D Loss: 13.582676 | G Loss: 1027.348877
[15/2000] D Loss: 12.538008 | G Loss: 1025.555542
[16/2000] D Loss: 11.555518 | G Loss: 1023.143372
[17/2000] D Loss: 10.624632 | G Loss: 1019.659729
[18/2000] D Loss: 9.750763 | G Loss: 1017.815979
[19/2000] D Loss: 8.940786 | G Loss: 1016.386230
[20/2000] D Loss: 8.179699 | G Loss: 1017.199951
[21/2000] D 

### c. Synthesize the datasets

In [12]:
df_syn_aux = synthesize(Q_aux_gen, gan_aux)
df_syn_target = synthesize(Q_target_gen, gan_target)

### d. Compute the threshold

In [13]:
n = 50
tids_to_remove_IN = np.random.choice(Q_aux_gen['tid'].unique(), size = (Q_aux_gen['tid'].nunique() - n), replace=False)
tids_to_remove_OUT = np.random.choice(Q_aux_test['tid'].unique(), size = (Q_aux_test['tid'].nunique() - n), replace=False)
IN_data = Q_aux_gen.copy()
OUT_data = Q_aux_test.copy()
IN_data = IN_data[~IN_data['tid'].isin(tids_to_remove_IN)]
OUT_data = OUT_data[~OUT_data['tid'].isin(tids_to_remove_OUT)]

In [14]:
IN_dist_aux = []
for tid in tqdm(IN_data['tid'].unique()):
    trajectory = IN_data.loc[IN_data['tid'] == tid]
    if Frechet:
        _, mini = find_closest_traj(df_syn_aux, trajectory) 
    elif Custom:
        _, mini = find_closest_custom(df_syn_aux, trajectory) 
    else:
        print('Error: metric not selected')
    IN_dist_aux.append(mini)

OUT_dist_aux = []
for tid in tqdm(OUT_data['tid'].unique()) :
    trajectory = OUT_data.loc[OUT_data['tid'] == tid]
    if Frechet:
        _, mini = find_closest_traj(df_syn_aux, trajectory) 
    elif Custom:
        _, mini = find_closest_custom(df_syn_aux, trajectory) 
    else:
        print('Error: metric not selected')
    OUT_dist_aux.append(mini)

100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:39<00:00,  1.27it/s]


In [15]:
# KDE estimation
kde_train = gaussian_kde(IN_dist_aux)
kde_test = gaussian_kde(OUT_dist_aux)

x_grid = np.linspace(min(min(IN_dist_aux), min(OUT_dist_aux)), max(max(IN_dist_aux), max(OUT_dist_aux)), 1000)

mode_train = x_grid[np.argmax(kde_train(x_grid))]
mode_test = x_grid[np.argmax(kde_test(x_grid))]

threshold = (mode_test - mode_train)/2 + mode_train

### e. Finish the attack and compute the accuracy

In [16]:
n = 50
tids_to_remove_IN = np.random.choice(Q_target_gen['tid'].unique(), size = (Q_target_gen['tid'].nunique() - n), replace=False)
tids_to_remove_OUT = np.random.choice(Q_target_test['tid'].unique(), size = (Q_target_test['tid'].nunique() - n), replace=False)
IN_data = Q_target_gen.copy()
OUT_data = Q_target_test.copy()
IN_data_target = IN_data[~IN_data['tid'].isin(tids_to_remove_IN)]
OUT_data_target = OUT_data[~OUT_data['tid'].isin(tids_to_remove_OUT)]

In [17]:
IN_dist_target = []
for tid in tqdm(IN_data_target['tid'].unique()):
    trajectory = IN_data_target.loc[IN_data_target['tid'] == tid]
    if Frechet:
        _, mini = find_closest_traj(df_syn_target, trajectory) 
    elif Custom:
        _, mini = find_closest_custom(df_syn_target, trajectory) 
    else:
        print('Error: metric not selected')
    IN_dist_target.append(mini)

OUT_dist_target = []
for tid in tqdm(OUT_data_target['tid'].unique()) :
    trajectory = OUT_data_target.loc[OUT_data_target['tid'] == tid]
    if Frechet:
        _, mini = find_closest_traj(df_syn_target, trajectory)
    elif Custom:
        _, mini = find_closest_custom(df_syn_target, trajectory)
    else:
        print('Error: metric not selected')
    OUT_dist_target.append(mini)

100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:39<00:00,  1.28it/s]


In [18]:
true_labels = []
predictions = []
for dist in OUT_dist_target:
    true_labels.append(0)
    predictions.append(1 if dist <= threshold else 0)
for dist in IN_dist_target:
    true_labels.append(1)
    predictions.append(1 if dist <= threshold else 0)

In [19]:
accuracy_score(true_labels, predictions)

0.99

In [20]:
SEED

2023

### Results Main FC

Tab for the Frechet metric, with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.87|
|2024|0.89|
|2025|0.87|
|2026|0.9|

Mean: 0.883 Std: 0.013

Tab for the Custom metric with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.99|
|2024|1.0|
|2025|0.98|
|2026|0.97|

Mean: 0.985 Std: 0.011

### Results Main Geolife


Tab for the Frechet metric, with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.82|
|2024|0.87|
|2025|0.88|
|2026|0.76|

Mean: 0.833 Std: 0.048

Tab for the Custom metric with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.94|
|2024|0.91|
|2025|0.98|
|2026|0.95|

Mean: 0.945 Std: 0.025

## 2. Masked MIA

### a. Utils

In [21]:
def mask(df, ratio):
    df_masked = df.copy().reset_index(drop=True)
    for tid in tqdm(df['tid'].unique()):
        trajectory = df_masked.loc[df_masked['tid'] == tid]
        mask_indices_tid = trajectory.sample(frac=ratio).index
        df_masked.loc[mask_indices_tid, ['lat','lon']] = np.nan
    return df_masked

def find_closest_traj_masked(df_syn, trajectory):
    # trajectory does not contain any tid and label column
    dis_euclide = []
    tid_min = None
    mini = 99
    tamp_lat = np.array(trajectory["lat"]).reshape(-1,1)
    tamp_lon = np.array(trajectory["lon"]).reshape(-1,1)
    real_traj = np.concatenate([tamp_lat, tamp_lon], axis= 1)
    # we retrieve the mask of the trajectory, while keeping the indices of the mask
    mask = np.isnan(real_traj[:, 0]) & np.isnan(real_traj[:, 1])
    indices_mask = np.where(mask)[0]
    real_traj_masked = np.delete(real_traj, indices_mask, axis=0)
    for label in df_syn['label'].unique():
        for tid in df_syn[df_syn['label']==label]['tid'].unique():
            syn_lat = np.array(df_syn.loc[df_syn['tid'] == tid]["lat"]).reshape(-1,1)
            syn_lon = np.array(df_syn.loc[df_syn['tid'] == tid]["lon"]).reshape(-1,1)
            
            if len(syn_lat) == len(tamp_lat) :
                syn_traj = np.concatenate([syn_lat, syn_lon], axis=1)
                # we mask the synthetic trajectory the same way, to keep consistency between the traces 
                syn_traj_masked = np.delete(syn_traj, indices_mask, axis=0)
                tamp = frdist(real_traj_masked, syn_traj_masked)
                if mini >= tamp:
                    mini = tamp
                    tid_min = tid
    return tid_min, mini

def find_closest_custom_masked(df_syn, trajectory):
    distances = []
    tid_min = None
    mini = 99
    tamp_lat = np.array(trajectory["lat"]).reshape(-1,1)
    tamp_lon = np.array(trajectory["lon"]).reshape(-1,1)
    real_traj = np.concatenate([tamp_lat, tamp_lon], axis= 1)
    # we retrieve the mask of the trajectory, while keeping the indices of the mask
    mask = np.isnan(real_traj[:, 0]) & np.isnan(real_traj[:, 1])
    indices_mask = np.where(mask)[0]
    real_traj_masked = np.delete(real_traj, indices_mask, axis=0)
    for label in df_syn['label'].unique():
        for tid in df_syn[df_syn['label']==label]['tid'].unique():
            syn_lat = np.array(df_syn.loc[df_syn['tid'] == tid]["lat"]).reshape(-1,1)
            syn_lon = np.array(df_syn.loc[df_syn['tid'] == tid]["lon"]).reshape(-1,1)
            syn_cat = np.array(df_syn.loc[df_syn['tid'] == tid]["category"]).reshape(1,-1)
            syn_hour = np.array(df_syn.loc[df_syn['tid'] == tid]["hour"]).reshape(1,-1)
            syn_day = np.array(df_syn.loc[df_syn['tid'] == tid]["day"]).reshape(1,-1)

            if len(syn_lat) == len(tamp_lat) :
                syn_cat_masked = np.delete(syn_cat, indices_mask, axis=1)
                syn_hour_masked = np.delete(syn_hour, indices_mask, axis=1)
                syn_day_masked = np.delete(syn_day, indices_mask, axis=1)
                syn_traj = np.concatenate([syn_lat, syn_lon], axis=1)
                syn_traj_masked = np.delete(syn_traj, indices_mask, axis=0)
                frech_dist = frdist(real_traj_masked, syn_traj_masked)
                real_cat_masked = np.delete(np.array(trajectory["category"]).reshape(1,-1), indices_mask, axis=1)
                real_hour_masked = np.delete(np.array(trajectory["hour"]).reshape(1,-1), indices_mask, axis=1)
                real_day_masked = np.delete(np.array(trajectory["day"]).reshape(1,-1), indices_mask, axis=1)
                cos_dist_cat = 1 - cosine_similarity(real_cat_masked, syn_cat_masked)[0][0]
                cos_dist_hour = 1 - cosine_similarity(real_hour_masked, syn_hour_masked)[0][0]
                cos_dist_day = 1 - cosine_similarity(real_day_masked, syn_day_masked)[0][0]
                tamp = cos_dist_cat + cos_dist_hour + cos_dist_day + frech_dist
                if mini >= tamp:
                    mini = tamp
                    tid_min = tid
    return tid_min, mini

### b. Mask application on the target data

In [22]:
# First, consider the parameter alpha to describe the purcentage of the trace we leave out
alpha = 0.75
# Second, remove the information from the targets
OUT_data_masked = mask(OUT_data_target, alpha)
IN_data_masked = mask(IN_data_target, alpha)

100%|█████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 649.76it/s]


In [23]:
OUT_dist_mask = []
for tid in tqdm(OUT_data_masked['tid'].unique()) :
    trajectory = OUT_data_masked.loc[OUT_data_masked['tid'] == tid]
    if Frechet:
        tid_min, mini = find_closest_traj_masked(df_syn_target, trajectory) 
    elif Custom:
        tid_min, mini = find_closest_custom_masked(df_syn_target, trajectory) 
    else:
        print('Error: metric not selected')
    OUT_dist_mask.append(mini)
IN_dist_mask = []
for tid in tqdm(IN_data_masked['tid'].unique()):
    trajectory = IN_data_masked.loc[IN_data_masked['tid'] == tid]
    if Frechet:
        tid_min, mini = find_closest_traj_masked(df_syn_target, trajectory) 
    elif Custom:
        tid_min, mini = find_closest_custom_masked(df_syn_target, trajectory) 
    else:
        print('Error: metric not selected')
    IN_dist_mask.append(mini)

100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:37<00:00,  1.32it/s]


In [24]:
true_labels = []
predictions_masked = []
for dist in OUT_dist_mask:
    true_labels.append(0)
    predictions_masked.append(1 if dist <= threshold else 0)
for dist in IN_dist_mask:
    true_labels.append(1)
    predictions_masked.append(1 if dist <= threshold else 0)

In [25]:
accuracy_score(true_labels, predictions_masked)

0.79

In [26]:
SEED

2023

### Results Mask FC

Tab for the Frechet metric, with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.81|
|2024|0.83|
|2025|0.86|
|2026|0.85|

Mean: 0.838 Std: 0.019

Tab for the Custom metric with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.79|
|2024|0.82|
|2025|0.81|
|2026|0.85|

Mean: 0.818 Std: 0.022

### Results Mask Geolife

Tab for the Frechet metric, with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.77|
|2024|0.82|
|2025|0.83|
|2026|0.75|

Mean: 0.793 Std: 0.033

Tab for the Custom metric with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.75|
|2024|0.77|
|2025|0.82|
|2026|0.82|

Mean: 0.790 Std: 0.031

## 3. Synthetic only MIA

In this specific setting, we now don't let access to the auxiliary information. 
Instead, we provide a greater synthetic dataset (by adding D_aux_train, Q_aux_gen and Q_aux_test to Q_target_gen).

To facilitate the comprehension of the attack, we keep the split, and we generate the synthetic information for each of them. 
The goal of the attacker will remain to infer the mebership in Q_target_gen, but with the use of D_aux_train_synth, Q_aux_gen_synth and Q_aux_test_synth

### a. Production of the synthetic information

In [27]:
df_syn_target = synthesize(Q_target_gen, gan_target)
df_aux_train_syn = synthesize(df_aux_train, gan_target)
Q_aux_gen_syn = synthesize(Q_aux_gen, gan_target)
Q_aux_test_syn = synthesize(Q_aux_test, gan_target)

### b. Recomputation of the new LSTM-TrajGAN based only on the synthetic information

In [28]:
# Global variables set by LSTMTrajGAN
latent_dim = 100
max_length = 144

keys = ['lat_lon', 'day', 'hour', 'category', 'mask']
vocab_size = {"lat_lon":2,"day":7,"hour":24,"category":10,"mask":1}

n_epochs = 2000
n_batch_size = 128
n_sample_interval = 100

lat_centroid_aux_syn = df_aux_train_syn['lat'].sum()/len(df_aux_train_syn)
lon_centroid_aux_syn = df_aux_train_syn['lon'].sum()/len(df_aux_train_syn)

scale_factor_aux_syn = max(max(abs(df_aux_train_syn['lat'].max() - lat_centroid_aux_syn),
                               abs(df_aux_train_syn['lat'].min() - lat_centroid_aux_syn),
                    ),
                           max(abs(df_aux_train_syn['lon'].max() - lon_centroid_aux_syn),
                               abs(df_aux_train_syn['lon'].min() - lon_centroid_aux_syn),
                            ))

gan_aux_syn = LSTM_TrajGAN(latent_dim, keys, vocab_size, max_length, lat_centroid_aux_syn, lon_centroid_aux_syn, scale_factor_aux_syn, SEED, all_params_path = GAN_PATH)
gan_aux_syn_path = f'{GAN_PATH}\training_params\G_model_' + str(n_epochs) + '_paper_auxiliary_synthetic_only.h5'

df_aux_train_syn_normalized = df_aux_train_syn.copy()
df_aux_train_syn_normalized["lat"] = df_aux_train_syn_normalized["lat"].apply(lambda x : x - lat_centroid_aux_syn)
df_aux_train_syn_normalized["lon"] = df_aux_train_syn_normalized["lon"].apply(lambda x : x - lon_centroid_aux_syn)
gan_aux_syn.train(train=df_aux_train_syn_normalized, epochs=n_epochs, batch_size=n_batch_size, sample_interval=n_sample_interval, adding = 'paper_auxiliary_synthetic_only')

[1/2000] D Loss: 32.335457 | G Loss: 1089.589478
[2/2000] D Loss: 30.705700 | G Loss: 1068.731567
[3/2000] D Loss: 29.050049 | G Loss: 1064.600098
[4/2000] D Loss: 27.363720 | G Loss: 1060.570679
[5/2000] D Loss: 25.713390 | G Loss: 1049.085693
[6/2000] D Loss: 24.120539 | G Loss: 1047.780029
[7/2000] D Loss: 22.580441 | G Loss: 1043.927734
[8/2000] D Loss: 21.101849 | G Loss: 1037.461182
[9/2000] D Loss: 19.686405 | G Loss: 1035.107910
[10/2000] D Loss: 18.344875 | G Loss: 1033.271851
[11/2000] D Loss: 17.054090 | G Loss: 1029.225586
[12/2000] D Loss: 15.824224 | G Loss: 1023.354736
[13/2000] D Loss: 14.663128 | G Loss: 1022.460754
[14/2000] D Loss: 13.560899 | G Loss: 1023.450195
[15/2000] D Loss: 12.523311 | G Loss: 1021.826843
[16/2000] D Loss: 11.537745 | G Loss: 1016.415833
[17/2000] D Loss: 10.609322 | G Loss: 1012.283508
[18/2000] D Loss: 9.732079 | G Loss: 1009.530640
[19/2000] D Loss: 8.924031 | G Loss: 1009.962891
[20/2000] D Loss: 8.168137 | G Loss: 1006.531982
[21/2000] D 

### c. Synthesize attacker datasets

In [29]:
df_syn_aux_synthetic_only = synthesize(Q_aux_gen_syn, gan_aux_syn)

### d. Compute the threshold using the synthetic information

In [30]:
n = 50
tids_to_remove_IN = np.random.choice(Q_aux_gen_syn['tid'].unique(), size = (Q_aux_gen_syn['tid'].nunique() - n), replace=False)
tids_to_remove_OUT = np.random.choice(Q_aux_test_syn['tid'].unique(), size = (Q_aux_test_syn['tid'].nunique() - n), replace=False)
IN_data_syn = Q_aux_gen_syn.copy()
OUT_data_syn = Q_aux_test_syn.copy()
IN_data_syn = IN_data_syn[~IN_data_syn['tid'].isin(tids_to_remove_IN)]
OUT_data_syn = OUT_data_syn[~OUT_data_syn['tid'].isin(tids_to_remove_OUT)]

In [31]:
IN_dist_aux_syn = []
for tid in tqdm(IN_data_syn['tid'].unique()):
    trajectory = IN_data_syn.loc[IN_data_syn['tid'] == tid]
    if Frechet:
        _, mini = find_closest_traj(df_syn_aux_synthetic_only, trajectory) 
    elif Custom:
        _, mini = find_closest_custom(df_syn_aux_synthetic_only, trajectory) 
    else:
        print('Error: metric not selected')
    IN_dist_aux_syn.append(mini)

OUT_dist_aux_syn = []
for tid in tqdm(OUT_data_syn['tid'].unique()) :
    trajectory = OUT_data_syn.loc[OUT_data_syn['tid'] == tid]
    if Frechet:
        _, mini = find_closest_traj(df_syn_aux_synthetic_only, trajectory) 
    elif Custom:
        _, mini = find_closest_custom(df_syn_aux_synthetic_only, trajectory)
    else:
        print('Error: metric not selected')
    OUT_dist_aux_syn.append(mini)

100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:38<00:00,  1.30it/s]


In [32]:
# KDE estimation
kde_train = gaussian_kde(IN_dist_aux_syn)
kde_test = gaussian_kde(OUT_dist_aux_syn)

x_grid = np.linspace(min(min(IN_dist_aux_syn), min(OUT_dist_aux_syn)), max(max(IN_dist_aux_syn), max(OUT_dist_aux_syn)), 1000)

mode_train = x_grid[np.argmax(kde_train(x_grid))]
mode_test = x_grid[np.argmax(kde_test(x_grid))]

threshold = mode_test - abs(mode_train - mode_test)/2

### e. Finish the synthetic only attack and compute the accuracy

In [33]:
n = 50
tids_to_remove_IN = np.random.choice(Q_target_gen['tid'].unique(), size = (Q_target_gen['tid'].nunique() - n), replace=False)
tids_to_remove_OUT = np.random.choice(Q_target_test['tid'].unique(), size = (Q_target_test['tid'].nunique() - n), replace=False)
IN_data = Q_target_gen.copy()
OUT_data = Q_target_test.copy()
IN_data_target = IN_data[~IN_data['tid'].isin(tids_to_remove_IN)]
OUT_data_target = OUT_data[~OUT_data['tid'].isin(tids_to_remove_OUT)]

In [34]:
IN_dist_target = []
for tid in tqdm(IN_data_target['tid'].unique()):
    trajectory = IN_data_target.loc[IN_data_target['tid'] == tid]
    if Frechet:
        _, mini = find_closest_traj(df_syn_target, trajectory) 
    elif Custom:
        _, mini = find_closest_custom(df_syn_target, trajectory)
    else:
        print('Error: metric not selected')
    IN_dist_target.append(mini)

OUT_dist_target = []
for tid in tqdm(OUT_data_target['tid'].unique()) :
    trajectory = OUT_data_target.loc[OUT_data_target['tid'] == tid]
    if Frechet:
        _, mini = find_closest_traj(df_syn_target, trajectory)
    elif Custom:
        _, mini = find_closest_custom(df_syn_target, trajectory)
    else:
        print('Error: metric not selected')
    OUT_dist_target.append(mini)

100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:38<00:00,  1.30it/s]


In [35]:
true_labels = []
predictions = []
for dist in OUT_dist_target:
    true_labels.append(0)
    predictions.append(1 if dist <= threshold else 0)
for dist in IN_dist_target:
    true_labels.append(1)
    predictions.append(1 if dist <= threshold else 0)

In [36]:
accuracy_score(true_labels, predictions)

0.99

In [37]:
SEED

2023

### Results Synthetic Only FC

Tab for the Frechet metric, with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.94|
|2024|0.91|
|2025|0.92|
|2026|0.93|

Mean: 0.925 Std: 0.011

Tab for the Custom metric with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.99|
|2024|0.96|
|2025|0.99|
|2026|1.0|

Mean: 0.985 Std: 0.015

### Results Synthetic Only Geolife

Tab for the Frechet metric, with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.86|
|2024|0.92|
|2025|0.89|
|2026|0.81|

Mean: 0.870 Std: 0.041

Tab for the Custom metric with 2000/2000:
| Seed | Accuracy score |
|----------|------------|
|2023|0.93|
|2024|0.94|
|2025|0.99|
|2026|0.94|

Mean: 0.950 Std: 0.023